# RNA-Seq Processing Pipeline
### Authors: Emily Skates, Stephen Williams

Processes STAR + featureCounts outputs through downstream analyses including DESeq2 differential expression, PCA/PERMANOVA clustering, WGCNA co-expression networks, and Gene Ontology/GSEA enrichment.

## 1. Environment Setup & Global Configuration
Initialises required libraries, sets the working directory, and defines global thresholds for gene filtering, plotting, and differential expression cut-offs.

In [ ]:
setwd("/Users/steve/Library/CloudStorage/GoogleDrive-stev3.w1l@gmail.com/My Drive/Academia/c_collaborations/emilyS/")

required_packages <- c("DESeq2", "ggplot2", "ggvenn", "WGCNA", "pheatmap", "vegan",
                       "pairwiseAdonis", "glue", "scales", "gtools", "clusterProfiler",
                       "enrichplot", "org.Mm.eg.db", "patchwork")

missing_packages <- required_packages[!(required_packages %in% installed.packages()[, "Package"])]
if (length(missing_packages)) {
    install.packages(missing_packages)
}

suppressPackageStartupMessages({
    lapply(required_packages, library, character.only = TRUE)
})

min_gene_copies           <- 10
min_samples_per_condition <- 1
num_pca_genes             <- 500
num_network_genes         <- 100
log2_fc_threshold         <- 1.5
adjusted_p_threshold      <- 0.05
nominal_p_threshold       <- 0.05

options(repr.plot.width = 8, repr.plot.height = 4) # Set default plot size for Jupyter notebooks

## 2. Data Loading & DESeq2 Initialisation
Compiles the individual gene count files into a unified matrix, defines the sample metadata (including the nested subject design), and initialises the `DESeqDataSet`.

### Understanding the Experimental Design Formula

The DESeq2 design formula breaks down the total variance in gene expression into specific biological, experimental, and technical components:

$$\text{Design Matrix} \sim \text{LabelingState} + \text{LabelingState:SubjectId} + \text{AssayType} + \text{LabelingState:AssayType}$$

* **`LabelingState`**: Controls for the baseline difference between the labeled and unlabeled conditions (e.g., global transcriptomic shifts caused by the metabolic labeling itself).
* **`LabelingState:SubjectId`**: Accounts for individual biological variation by nesting the samples strictly within their respective labeling groups. This captures the paired relationship between an individual sample's Input and Pull-Down samples, preventing sample-to-sample baseline differences from muddying the results.
* **`AssayType`**: Measures the non-specific background enrichment. It captures the genes that naturally stick to the beads or tube during a pull-down even when no label is present:
  $$\text{Unlabeled Pull-Down} - \text{Unlabeled Input}$$
* **`LabelingState:AssayType`**: The interaction term between label/pulldown state that isolates the **True Enrichment**. It calculates the "difference of differences," subtracting non-specific background stickiness from the labeled enrichment to find genes specifically captured by the assay:

$$\text{True Target Enrichment} = (\text{Labeled Pull-Down} - \text{Labeled Input}) - (\text{Unlabeled Pull-Down} - \text{Unlabeled Input})$$

In [ ]:
count_files <- mixedsort(Sys.glob("o_outputs/sample_*/gene_counts.txt"))

initial_file_data <- read.table(count_files[1], header = TRUE, skip = 1, stringsAsFactors = FALSE)
unified_count_matrix <- data.frame(Geneid = initial_file_data[, 1])

for (file_path in count_files) {
    sample_id <- basename(dirname(file_path))
    sample_data <- read.table(file_path, header = TRUE, skip = 1, stringsAsFactors = FALSE)
    unified_count_matrix[[sample_id]] <- sample_data[, 7]
}

row.names(unified_count_matrix) <- unified_count_matrix$Geneid
unified_count_matrix$Geneid <- NULL

experimental_metadata <- data.frame(
    LabelingState = factor(rep(c("unlab", "lab", "unlab", "lab"), each = 3), levels = c("unlab", "lab")),
    AssayType = factor(rep(c("pull_down", "input"), each = 6), levels = c("input", "pull_down")),
    SubjectId = factor(rep(c("Subj1", "Subj2", "Subj3"), times = 4))
)
row.names(experimental_metadata) <- colnames(unified_count_matrix)

deseq_dataset <- DESeqDataSetFromMatrix(
    countData = unified_count_matrix,
    colData = experimental_metadata,
    design = ~ LabelingState + LabelingState:SubjectId + AssayType + LabelingState:AssayType
)

genes_to_keep <- rowSums(counts(deseq_dataset) >= min_gene_copies) >= min_samples_per_condition
deseq_dataset <- deseq_dataset[genes_to_keep, ]
vst_transformed_data <- vst(deseq_dataset, blind = FALSE)

## 3. Dispersion Diagnostics
Calculates size factors and dispersion estimates, plotting the fit to ensure data quality and model appropriateness.

### Understanding DESeq2 Dispersion Plots

In RNA-Seq data, gene expression variance naturally scales with the mean (highly expressed genes show higher absolute variance). 
To model this accurately, DESeq2 estimates a **dispersion** parameter for each gene, which measures how much a gene's variance exceeds its mean.

A standard diagnostics plot visualises this relationship across three layers:
1. **Black Dots (`dispGeneEst`)**: The raw, independent dispersion estimate for each individual gene based purely on its sample variance.
2. **Red Line (`dispFit`)**: The global trend line fit across all genes, capturing how dispersion typically decreases as the mean expression increases.
3. **Blue Dots (`dispersion`)**: The final, shrunken dispersion values used for differential expression testing. DESeq2 shrinks gene-wise estimates toward the red line to minimise false positives from low-abundance genes, while allowing true biological outliers to remain high.

The horizontal flat "ceiling" of blue and black dots, present at the top-left corner of the plot, corresponds to low-expression genes with extremely high variance.
Because these genes have extremely low mean counts, they are expected to naturally be filtered out or mathematically penalised during differential expression testing due to their massive variance. They should not skew your downstream results or cause false positives. 

In [ ]:
deseq_dataset <- estimateSizeFactors(deseq_dataset, quiet = TRUE)
deseq_dataset <- estimateDispersions(deseq_dataset, quiet = TRUE)

dispersion_metrics <- data.frame(
    baseMean    = mcols(deseq_dataset)$baseMean,
    dispGeneEst = mcols(deseq_dataset)$dispGeneEst,
    dispersion  = mcols(deseq_dataset)$dispersion,
    dispFit     = mcols(deseq_dataset)$dispFit
)
dispersion_metrics <- dispersion_metrics[dispersion_metrics$baseMean > 0, ]

dispersion_plot <- ggplot(dispersion_metrics, aes(x = baseMean)) +
    geom_point(aes(y = dispGeneEst), color = "black", alpha = 0.3, size = 1) +
    geom_point(aes(y = dispersion), color = "dodgerblue", alpha = 0.3, size = 1, shape = 1) +
    geom_line(aes(y = dispFit), color = "red", linewidth = 1) +
    scale_x_log10() + scale_y_log10() +
    theme_minimal() +
    labs(title = "Dispersion Estimates", x = "Mean of Normalised Counts", y = "Dispersion")

print(dispersion_plot)
ggsave("f_figures/Dispersion_Estimates.png", plot = dispersion_plot, width = 8, height = 6, dpi = 300)

## 4. Dimensionality Reduction & Diversity Testing
Performs Principal Component Analysis (PCA) to check sample clustering, followed by PERMANOVA and Beta Dispersion tests.

In [ ]:

pca_results <- plotPCA(vst_transformed_data, intgroup = c("LabelingState", "AssayType"), ntop = num_pca_genes, returnData = TRUE)

pca_palette <- c("lab:input" = "#D55E00", "lab:pull_down" = "#009E73",
                 "unlab:input" = "#56B4E9", "unlab:pull_down" = "#CC79A7")
pca_shapes <- c("lab:input" = 16, "lab:pull_down" = 17,
                "unlab:input" = 15, "unlab:pull_down" = 18)
pca_labels <- c("lab:input" = "Labeled Input", "lab:pull_down" = "Labeled Pull-Down",
                "unlab:input" = "Unlabeled Input", "unlab:pull_down" = "Unlabeled Pull-Down")

variance_explained <- round(attr(pca_results, "percentVar") * 100, 1)

pca_figure <- ggplot(pca_results, aes(x = PC1, y = PC2, color = group, shape = group)) +
    geom_point(size = 2.5) +
    scale_color_manual(name = "Group", values = pca_palette, labels = pca_labels) +
    scale_shape_manual(name = "Group", values = pca_shapes, labels = pca_labels) +
    labs(x = glue("PC1: {variance_explained[1]}% variance"),
         y = glue("PC2: {variance_explained[2]}% variance")) +
    theme_minimal() +
    theme(panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5))

print(pca_figure)

distance_matrix <- vegdist(t(assay(vst_transformed_data)), method = "euclidean")

print("--- Multi-Factor PERMANOVA (LabelingState * AssayType) ---")
print(adonis2(distance_matrix ~ LabelingState * AssayType, data = experimental_metadata, permutations = 999))

print("--- Single Factor PERMANOVA (AssayType) ---")
print(adonis2(distance_matrix ~ AssayType, data = experimental_metadata, permutations = 999))

print("--- Beta Dispersion ---")
interaction_groups <- interaction(experimental_metadata$LabelingState, experimental_metadata$AssayType)
print(anova(betadisper(distance_matrix, group = interaction_groups)))

## 5. Differential Expression Analysis
Executes the DESeq model, extracts relevant contrasts using LFC shrinkage, and categorises genes into strict confidence tiers.

### Differential Expression Stratification & Tiered Categorisation

To isolate biological signals from technical background noise, the pipeline evaluates three distinct contrasts using DESeq2:

1. **`Lab_PD_vs_Input`**: Evaluates enrichment in the labeled pull-down sample against its paired labeled input background.
2. **`Lab_PD_vs_All_Input`**: Compares the labeled pull-down against a pooled average of both labeled and unlabeled inputs, providing a robust baseline control.
3. **`Lab_vs_Unlab_PD`**: Compares the labeled pull-down directly against the unlabeled pull-down to identify transcripts whose capture is strictly dependent on the presence of the metabolic label.

#### Confidence Tiering Criteria

Post-shrinkage log₂ fold changes ($L2FC$) and significance statistics (adjusted p-values/$qval$ and nominal p-values/$pval$) are combined to group genes into strict confidence tiers based on the directionality and strength of their expression shifts:

Our strong confidence criteria is based on: Subcellular transcriptome sequencing with single cell APEX-seq identifies regulators of cell-cell interactions
Andrew Xue, Bo Cai, Qian Xue, Nianping Liu, Xiaojie Qiu, Rogelio A. Hernández-López, Alice Y. Ting
bioRxiv 2026.03.17.712496; doi: https://doi.org/10.64898/2026.03.17.712496

* **Tier 1 (Strongest Confidence)**: Significant after multiple testing corrections with a substantial fold change.
    * *Up*: $\log_2\text{(Fold Change)} > 0.5$ and $\text{Adjusted p-value} < 0.05$
    * *Down*: $\log_2\text{(Fold Change)} < -0.5$ and $\text{Adjusted p-value} < 0.05$
* **Tier 2 (Moderate Confidence)**: Significant after multiple testing corrections but displaying a milder directional shift.
    * *Up*: $0.0 < \log_2\text{(Fold Change)} \le 0.5$ and $\text{Adjusted p-value} < 0.05$
    * *Down*: $-0.5 \le \log_2\text{(Fold Change)} < 0.0$ and $\text{Adjusted p-value} < 0.05$
* **Tier 3 (Weak Stats, High Magnitude)**: Fails multiple testing corrections but exhibits a sharp change in expression magnitude with a significant unadjusted p-value.
    * *Up*: $\log_2\text{(Fold Change)} > 0.5$ and $\text{Nominal p-value} < 0.05$
    * *Down*: $\log_2\text{(Fold Change)} < -0.5$ and $\text{Nominal p-value} < 0.05$
* **Tier 4 (Weakest Confidence)**: Fails multiple testing corrections and shows a mild directional shift, backed only by a significant unadjusted p-value.
    * *Up*: $0.0 < \log_2\text{(Fold Change)} \le 0.5$ and $\text{Nominal p-value} < 0.05$
    * *Down*: $-0.5 \le \log_2\text{(Fold Change)} < 0.0$ and $\text{Nominal p-value} < 0.05$
* **Not Significant**: Genes failing to meet the minimum threshold criteria ($\text{Nominal p-value} \ge 0.05$).

In [ ]:
deseq_dataset <- DESeq(deseq_dataset, quiet = TRUE)
model_coefficients <- resultsNames(deseq_dataset)

contrast_labeled_pd_vs_input <- list(c("AssayType_pull_down_vs_input", "LabelingStatelab.AssayTypepull_down"))
de_labeled_pd_vs_input <- results(deseq_dataset, contrast = contrast_labeled_pd_vs_input)
shrunk_labeled_pd_vs_input <- lfcShrink(deseq_dataset, contrast = contrast_labeled_pd_vs_input, type = "ashr", quiet = TRUE)

contrast_labeled_pd_vs_all_input <- rep(0, length(model_coefficients))
contrast_labeled_pd_vs_all_input[which(model_coefficients == "AssayType_pull_down_vs_input")] <- 1
contrast_labeled_pd_vs_all_input[which(model_coefficients == "LabelingStatelab.AssayTypepull_down")] <- 0.5
de_labeled_pd_vs_all_input <- results(deseq_dataset, contrast = contrast_labeled_pd_vs_all_input)
shrunk_labeled_pd_vs_all_input <- lfcShrink(deseq_dataset, contrast = contrast_labeled_pd_vs_all_input, type = "ashr", quiet = TRUE)

contrast_labeled_vs_unlabeled_pd <- list(c("LabelingState_lab_vs_unlab", "LabelingStatelab.AssayTypepull_down"))
de_labeled_vs_unlabeled_pd <- results(deseq_dataset, contrast = contrast_labeled_vs_unlabeled_pd)
shrunk_labeled_vs_unlabeled_pd <- lfcShrink(deseq_dataset, contrast = contrast_labeled_vs_unlabeled_pd, type = "ashr", quiet = TRUE)

categorise_genes <- function(unshrunk_data, shrunk_data, fc_cut, padj_cut, pval_cut) {
    results_df <- data.frame(
        log2FoldChange = shrunk_data$log2FoldChange,
        padj = unshrunk_data$padj,
        pvalue = unshrunk_data$pvalue,
        row.names = rownames(shrunk_data)
    )
    results_df <- results_df[!is.na(results_df$padj) & !is.na(results_df$pvalue), ]

    is_highly_upregulated <- results_df$log2FoldChange > fc_cut
    is_moderately_upregulated <- results_df$log2FoldChange > 0 & results_df$log2FoldChange <= fc_cut
    is_highly_downregulated <- results_df$log2FoldChange < -fc_cut
    is_moderately_downregulated <- results_df$log2FoldChange >= -fc_cut & results_df$log2FoldChange < 0

    has_strong_significance <- results_df$padj < padj_cut
    has_weak_significance <- results_df$padj >= padj_cut & results_df$pvalue < pval_cut

    results_df$Confidence_Tier <- "Not Significant"
    results_df$Confidence_Tier[is_highly_upregulated & has_strong_significance]       <- "Tier 1: L2FC>0.5, qval<0.05"
    results_df$Confidence_Tier[is_moderately_upregulated & has_strong_significance]   <- "Tier 2: L2FC>0.0, qval<0.05"
    results_df$Confidence_Tier[is_highly_upregulated & has_weak_significance]         <- "Tier 3: L2FC>0.5, pval<0.05"
    results_df$Confidence_Tier[is_moderately_upregulated & has_weak_significance]     <- "Tier 4: L2FC>0.0, pval<0.05"
    results_df$Confidence_Tier[is_highly_downregulated & has_strong_significance]     <- "Tier 1: L2FC<-0.5, qval<0.05"
    results_df$Confidence_Tier[is_moderately_downregulated & has_strong_significance] <- "Tier 2: L2FC<0.0, qval<0.05"
    results_df$Confidence_Tier[is_highly_downregulated & has_weak_significance]       <- "Tier 3: L2FC<-0.5, pval<0.05"
    results_df$Confidence_Tier[is_moderately_downregulated & has_weak_significance]   <- "Tier 4: L2FC<0.0, pval<0.05"

    all_possible_tiers <- c(
        "Tier 1: L2FC>0.5, qval<0.05",
        "Tier 2: L2FC>0.0, qval<0.05",
        "Tier 3: L2FC>0.5, pval<0.05",
        "Tier 4: L2FC>0.0, pval<0.05",
        "Tier 1: L2FC<-0.5, qval<0.05",
        "Tier 2: L2FC<0.0, qval<0.05",
        "Tier 3: L2FC<-0.5, pval<0.05",
        "Tier 4: L2FC<0.0, pval<0.05",
        "Not Significant"
    )
    
    results_df$Confidence_Tier <- factor(results_df$Confidence_Tier, levels = all_possible_tiers)
    
    return(results_df)
}

de_comparisons <- list(
    "Lab_PD_vs_Input"     = list(unshrunk = de_labeled_pd_vs_input, shrunk = shrunk_labeled_pd_vs_input),
    "Lab_PD_vs_All_Input" = list(unshrunk = de_labeled_pd_vs_all_input, shrunk = shrunk_labeled_pd_vs_all_input),
    "Lab_vs_Unlab_PD"     = list(unshrunk = de_labeled_vs_unlabeled_pd, shrunk = shrunk_labeled_vs_unlabeled_pd)
)

categorised_results <- list()
for (comparison_id in names(de_comparisons)) {
    message(glue("\n--- Categorising Comparison: {comparison_id} ---"))
    tier_df <- categorise_genes(
        de_comparisons[[comparison_id]]$unshrunk,
        de_comparisons[[comparison_id]]$shrunk,
        log2_fc_threshold,
        adjusted_p_threshold,
        nominal_p_threshold
    )
    print(table(tier_df$Confidence_Tier))
    categorised_results[[comparison_id]] <- tier_df
    write.csv(tier_df, file = glue("o_outputs/processed_data/DE_results/Tiered_Results_{comparison_id}.csv"))
}

tier_labeled_pd_vs_input <- categorised_results[["Lab_PD_vs_Input"]]
tier_labeled_pd_vs_all_input <- categorised_results[["Lab_PD_vs_All_Input"]]
tier_labeled_vs_unlabeled_pd <- categorised_results[["Lab_vs_Unlab_PD"]]

## 6. Differential Expression Visualisation (Volcano & MA Plots)

This section maps a custom diagnostic plotting function across all three shrunken DESeq2 contrasts. 

For each comparison, the workflow systematically evaluates log₂ fold change and multiple-testing adjusted p-value boundaries, producing two standard visual outputs:
1. **Volcano Plots**: Dissect biological statistical significance against structural expression magnitude shifts, colour-coding significantly changing transcripts.
2. **MA Plots (Bland-Altman plots)**: Scale the distribution of shrunken log₂ fold changes against the mean of normalised counts to verify that variation balances correctly across low, moderate, and high expression thresholds.

Outputs are printed directly inline and automatically exported using the canonical contrast name identifier keys.

In [ ]:
generate_diagnostic_plots <- function(contrast_results, contrast_name, fc_cut, padj_cut, ptype) {
    
    # 1. Prepare Data
    plot_data <- as.data.frame(contrast_results)
    plot_data <- plot_data[!is.na(plot_data$padj) & !is.na(plot_data$baseMean), ]
    
    plot_data$Significance <- "Not Significant"
    
    # Determine confidence terminology and filter logic based on ptype
    if (ptype == "pval") {
        confidence_label <- "Weak Confidence (Nominal p-value)"
        volcano_y_label  <- "-Log10(Nominal P-value)"
        
        plot_data$Significance[plot_data$log2FoldChange > fc_cut & plot_data$pvalue < padj_cut] <- "Up-regulated"
        plot_data$Significance[plot_data$log2FoldChange < -fc_cut & plot_data$pvalue < padj_cut] <- "Down-regulated"
        
        # Match volcano plot's y-axis tracking metric to the unadjusted p-value
        volcano_y_metric <- -log10(plot_data$pvalue)
    } else {
        confidence_label <- "High Confidence (Adjusted q-value)"
        volcano_y_label  <- "-Log10(Adjusted P-value)"
        
        plot_data$Significance[plot_data$log2FoldChange > fc_cut & plot_data$padj < padj_cut] <- "Up-regulated"
        plot_data$Significance[plot_data$log2FoldChange < -fc_cut & plot_data$padj < padj_cut] <- "Down-regulated"
        
        volcano_y_metric <- -log10(plot_data$padj)
    }
    
    # 2. Build Volcano Plot
    volcano_plot <- ggplot(plot_data, aes(x = log2FoldChange, y = volcano_y_metric, color = Significance)) +
        geom_point(alpha = 0.6, size = 1.2) + 
        scale_color_manual(values = c("Up-regulated" = "red", "Down-regulated" = "blue", "Not Significant" = "grey")) +
        geom_vline(xintercept = c(-fc_cut, fc_cut), linetype = "dashed", color = "black") +
        geom_hline(yintercept = -log10(padj_cut), linetype = "dashed", color = "black") +
        theme_minimal() +
        theme(legend.position = "none") + 
        labs(title = "Volcano Plot", x = "Log2 Fold Change", y = volcano_y_label)
    
    # 3. Build MA Plot
    ma_plot <- ggplot(plot_data, aes(x = baseMean, y = log2FoldChange, color = Significance)) +
        geom_point(alpha = 0.6, size = 1.2) +
        scale_x_log10(labels = scales::trans_format("log10", scales::math_format(10^.x))) +
        scale_color_manual(values = c("Up-regulated" = "red", "Down-regulated" = "blue", "Not Significant" = "grey")) +
        geom_hline(yintercept = 0, linetype = "solid", color = "black", linewidth = 0.5) +
        geom_hline(yintercept = c(-fc_cut, fc_cut), linetype = "dashed", color = "grey40") +
        coord_cartesian(ylim = c(-4, 4)) +
        theme_minimal() +
        labs(title = "MA Plot", x = "Mean of Normalised Counts", y = "Log2 Fold Change")
    
    # 4. Combine Plots Side-by-Side using Patchwork
    # The header title and confidence thresholds are now bound directly inside the plot canvas object
    combined_panel <- (volcano_plot + ma_plot) + 
        plot_layout(guides = "collect") + 
        plot_annotation(
            title    = glue("Diagnostic Plots: {contrast_name}"),
            subtitle = glue("Confidence Level: {confidence_label}  |  Thresholds: |L2FC| > {fc_cut}, p < {padj_cut}"),
            theme    = theme(
                plot.title    = element_text(size = 14, face = "bold", hjust = 0.5),
                plot.subtitle = element_text(size = 11, face = "italic", color = "grey30", hjust = 0.5)
            )
        )
    
    # 5. Print and Save the Unified Panel
    print(combined_panel)
    ggsave(glue("f_figures/DE/Diagnostic_Panel_{contrast_name}_{ptype}.png"), plot = combined_panel, width = 12, height = 5, dpi = 300)
}

# Organise shrunken contrasts into a structured, named list
shrunk_contrasts <- list(
    "Lab_PD_vs_Input"     = shrunk_labeled_pd_vs_input,
    "Lab_PD_vs_All_Input" = shrunk_labeled_pd_vs_all_input,
    "Lab_vs_Unlab_PD"     = shrunk_labeled_vs_unlabeled_pd
)

# Run execution loops (headers inside messages are removed to ensure zero console output fragmentation)
for (contrast_title in names(shrunk_contrasts)) {
    generate_diagnostic_plots(
        contrast_results = shrunk_contrasts[[contrast_title]], 
        contrast_name    = contrast_title, 
        fc_cut           = log2_fc_threshold, 
        padj_cut         = adjusted_p_threshold,
        ptype            = "padj"
    )
}

for (contrast_title in names(shrunk_contrasts)) {
    generate_diagnostic_plots(
        contrast_results = shrunk_contrasts[[contrast_title]], 
        contrast_name    = contrast_title, 
        fc_cut           = log2_fc_threshold, 
        padj_cut         = nominal_p_threshold,
        ptype            = "pval"
    )
}


## 7. Network Analysis (Gene Correlation)
Builds a co-expression network map of the most highly variable genes across all samples using Pearson correlation.

In [ ]:
# 1. Subset the matrix and convert it to a long data frame
heatmap_data <- normalised_counts[most_variable_genes, ] %>%
    as.data.frame() %>%
    tibble::rownames_to_column(var = "Gene") %>%
    pivot_longer(cols = -Gene, names_to = "Sample", values_to = "Expression")

# 2. (Optional) Join with metadata if you want to order or facet by experimental groups
heatmap_data <- heatmap_data %>% left_join(experimental_metadata_long, by = "Sample")

# 3. Create the heatmap using ggplot2
gg_heatmap <- ggplot(heatmap_data, aes(x = Sample, y = Gene, fill = Expression)) +
    geom_tile() +
    # Use a continuous colour scale (viridis is excellent for heatmaps)
    scale_fill_viridis_c(option = "magma", name = "VST Counts") + 
    labs(
        title = glue("Co-expression of Top {num_network_genes} Variable Genes"),
        x = "Samples",
        y = "Genes"
    ) +
    theme_minimal() +
    theme(
        axis.text.y = element_blank(), # Hides gene names just like show_rownames = FALSE
        axis.text.x = element_text(angle = 45, hjust = 1), # Tilts sample names for readability
        panel.grid = element_blank()
    )

# 4. Display the plot in the notebook
print(gg_heatmap)

# 5. Save the plot to disk
ggsave(
    filename = glue("f_figures/Network-coexpr/Top_{num_network_genes}_Coexpression_Heatmap.png"),
    plot = gg_heatmap,
    width = 8, height = 6, dpi = 300
)

## 8. Gene Ontology (GO) Enrichment Analysis & Visualisation
Identifies enriched biological processes among target gene clusters and produces dotplot visualisations.

In [ ]:
active_de_dataset <- tier_labeled_pd_vs_all_input
active_comparison_name <- "Lab_PD_vs_All_Input"

extract_genes_by_tier <- function(dataset, valid_tiers) {
    rownames(dataset[dataset$Confidence_Tier %in% valid_tiers, ])
}

genes_high_up <- extract_genes_by_tier(active_de_dataset, c("Tier 1: L2FC>0.5, qval<0.05", "Tier 2: L2FC>0.0, pval<0.05"))
genes_high_down <- extract_genes_by_tier(active_de_dataset, c("Tier 1: L2FC<-0.5, qval<0.05", "Tier 2: L2FC<0.0, qval<0.05"))
genes_weak_up <- extract_genes_by_tier(active_de_dataset, c("Tier 3: L2FC>0.5, pval<0.05", "Tier 4: L2FC>0.0, pval<0.05"))
genes_weak_down <- extract_genes_by_tier(active_de_dataset, c("Tier 3: L2FC<-0.5, pval<0.05", "Tier 4: L2FC<0.0, pval<0.05"))

execute_go_enrichment <- function(target_genes, background_universe) {
    if (length(target_genes) == 0) return(NULL)
    enrichGO(gene = target_genes, universe = background_universe, OrgDb = org.Mm.eg.db,
             keyType = "SYMBOL", ont = "ALL", pAdjustMethod = "BH",
             pvalueCutoff = 0.05, qvalueCutoff = 0.2, readable = TRUE)
}

assess_cilia_terms <- function(go_results_df, analysis_label) {
    if (nrow(go_results_df) == 0) return(message(glue("No enriched GO terms for {analysis_label}.")))
    cilia_matches <- go_results_df[grep("cili", go_results_df$Description, ignore.case = TRUE), ]
    message(glue("Cilium-related terms in {analysis_label}: {nrow(cilia_matches)}"))
    if (nrow(cilia_matches) > 0) {
        unique_genes <- sort(unique(unlist(strsplit(paste(cilia_matches$geneID, collapse = "/"), "/"))))
        message(glue("Found {length(unique_genes)} unique cilia genes. Sample: "), paste(head(unique_genes), collapse=", "))
    }
}

generate_go_dotplot <- function(go_object, plot_title, output_filename) {
    if (is.null(go_object) || nrow(go_object) == 0) return()
    dotplot_figure <- dotplot(go_object, showCategory = 20) +
        ggtitle(plot_title) +
        theme(axis.text.y = element_text(size = 10), axis.title.y = element_blank())
    print(dotplot_figure)
    ggsave(output_filename, plot = dotplot_figure, width = 10, height = 8, dpi = 300)
}

gene_universe <- rownames(active_de_dataset)
go_res_high_up   <- execute_go_enrichment(genes_high_up, gene_universe)
go_res_high_down <- execute_go_enrichment(genes_high_down, gene_universe)
go_res_weak_up   <- execute_go_enrichment(genes_weak_up, gene_universe)
go_res_weak_down <- execute_go_enrichment(genes_weak_down, gene_universe)

if (!is.null(go_res_high_up)) write.csv(as.data.frame(go_res_high_up), "o_outputs/processed_data/GO_results/GO_HighConf_UP.csv")
if (!is.null(go_res_high_down)) write.csv(as.data.frame(go_res_high_down), "o_outputs/processed_data/GO_results/GO_HighConf_DOWN.csv")
if (!is.null(go_res_weak_up)) write.csv(as.data.frame(go_res_weak_up), "o_outputs/processed_data/GO_results/GO_WeakConf_UP.csv")
if (!is.null(go_res_weak_down)) write.csv(as.data.frame(go_res_weak_down), "o_outputs/processed_data/GO_results/GO_WeakConf_DOWN.csv")

assess_cilia_terms(if (!is.null(go_res_high_up)) as.data.frame(go_res_high_up) else data.frame(), "High-Confidence Up Targets")
generate_go_dotplot(go_res_high_up, glue("Top Processes: Up-regulated in {active_comparison_name}"), glue("f_figures/GO/GO_UP_{active_comparison_name}.png"))
generate_go_dotplot(go_res_high_down, glue("Top Processes: Down-regulated in {active_comparison_name}"), glue("f_figures/GO/GO_DOWN_{active_comparison_name}.png"))

assess_cilia_terms(if (!is.null(go_res_weak_up)) as.data.frame(go_res_weak_up) else data.frame(), "Weak-Confidence Up Targets")
generate_go_dotplot(go_res_weak_up, glue("Top Processes: Weakly Up-regulated in {active_comparison_name}"), glue("f_figures/GO/GO_Weak_UP_{active_comparison_name}.png"))
generate_go_dotplot(go_res_weak_down, glue("Top Processes: Weakly Down-regulated in {active_comparison_name}"), glue("f_figures/GO/GO_Weak_DOWN_{active_comparison_name}.png"))

## 9. Gene Set Enrichment Analysis (GSEA)
Generates a ranked metric for genes to evaluate broader pathway activation/suppression, outputting global visualisations and targeted running scores.

In [ ]:
gsea_contrast_results <- shrunk_labeled_pd_vs_all_input
gsea_data <- as.data.frame(gsea_contrast_results)
gsea_data <- gsea_data[!is.na(gsea_data$pvalue) & !is.na(gsea_data$log2FoldChange), ]

gene_ranking_metric <- gsea_data$log2FoldChange / gsea_data$lfcSE
names(gene_ranking_metric) <- rownames(gsea_data)
gene_ranking_metric <- sort(gene_ranking_metric[!is.na(gene_ranking_metric)], decreasing = TRUE)

gsea_results_obj <- gseGO(geneList = gene_ranking_metric, OrgDb = org.Mm.eg.db, keyType = "SYMBOL",
                          ont = "ALL", pvalueCutoff = 0.1, pAdjustMethod = "BH", eps = 0, seed = 123)

if (!is.null(gsea_results_obj) && nrow(gsea_results_obj) > 0) {
    gsea_df <- as.data.frame(gsea_results_obj)
    write.csv(gsea_df, "o_outputs/processed_data/GO_results/GSEA_GO_Enrichment.csv")

    gsea_cilia_terms <- gsea_df[grep("cili", gsea_df$Description, ignore.case = TRUE), ]
    message(glue("GSEA Cilia pathways identified: {nrow(gsea_cilia_terms)}"))

    gsea_dotplot <- dotplot(gsea_results_obj, showCategory = 15, split = ".sign") +
        facet_grid(. ~ .sign) +
        ggtitle("GSEA Functional Shifts") +
        theme(axis.text.y = element_text(size = 8))

    print(gsea_dotplot)
    ggsave("f_figures/GO/GSEA_Dotplot.png", plot = gsea_dotplot, width = 11, height = 8, dpi = 300)

    if (nrow(gsea_cilia_terms) > 0) {
        gsea_mountain_plot <- gseaplot2(gsea_results_obj, geneSetID = gsea_cilia_terms$ID[1],
                                        title = glue("Running Enrichment: {gsea_cilia_terms$Description[1]}"))
        print(gsea_mountain_plot)
        ggsave("f_figures/GO/GSEA_Mountain_Plot_Cilia.png", plot = gsea_mountain_plot, width = 8, height = 6, dpi = 300)
    }
}

## 10. Deep-Dive: Ciliary Basal Body Bi-directional Drivers
Extracts "leading edge" genes from the GSEA mountain plot to identify specific drivers of bi-directional pathway shifts. Visualises expression, builds isolated co-expression networks, and exports lists for motif analysis.

In [ ]:
if (exists("gsea_cilia_terms") && nrow(gsea_cilia_terms) > 0) {

    primary_term_id <- gsea_cilia_terms$ID[1]
    primary_term_desc <- gsea_cilia_terms$Description[1]
    sanitised_term_name <- gsub(" ", "_", primary_term_desc)

    message(glue("\n--- Analysing Core Drivers for: {primary_term_desc} ({primary_term_id}) ---"))

    leading_edge_string <- gsea_cilia_terms$core_enrichment[1]
    leading_edge_genes <- unlist(strsplit(leading_edge_string, "/"))

    gene_lfc_values <- gsea_data[leading_edge_genes, "log2FoldChange"]
    names(gene_lfc_values) <- leading_edge_genes

    upregulated_drivers <- names(gene_lfc_values[gene_lfc_values > 0])
    downregulated_drivers <- names(gene_lfc_values[gene_lfc_values < 0])

    message(glue("Total Leading Edge Genes: {length(leading_edge_genes)}"))
    message(glue("Up-regulated Drivers: {length(upregulated_drivers)}"))
    message(glue("Down-regulated Drivers: {length(downregulated_drivers)}"))

    write.table(upregulated_drivers, glue("o_outputs/processed_data/GO_results/{sanitised_term_name}_Up_Drivers.txt"),
                row.names = FALSE, col.names = FALSE, quote = FALSE)
    write.table(downregulated_drivers, glue("o_outputs/processed_data/GO_results/{sanitised_term_name}_Down_Drivers.txt"),
                row.names = FALSE, col.names = FALSE, quote = FALSE)

    core_driver_counts <- normalised_counts[leading_edge_genes, ]

    pheatmap(core_driver_counts,
             annotation_col = experimental_metadata,
             scale = "row",
             show_rownames = TRUE,
             fontsize_row = 6,
             main = glue("Expression of {primary_term_desc} Leading Edge Drivers"),
             filename = glue("f_figures/GO/{sanitised_term_name}_Expression_Heatmap.png"),
             width = 8, height = 10, silent = FALSE)

    core_driver_correlation <- cor(t(core_driver_counts), method = "pearson")

    pheatmap(core_driver_correlation,
             show_rownames = TRUE,
             show_colnames = TRUE,
             fontsize_row = 6,
             fontsize_col = 6,
             main = glue("Co-expression Network: {primary_term_desc}"),
             filename = glue("f_figures/Network-coexpr/{sanitised_term_name}_Coexpression_Heatmap.png"),
             width = 10, height = 10, silent = FALSE)

    message("Analysis complete. Targeted heatmaps and separated driver lists have been exported.")

} else {
    message("No cilia terms found in GSEA results to analyse.")
}